# Extracción de Firma de Hoja (F_h)
Compila la abundancia de área y los descriptores morfológicos para generar una firma única representativa de la página.

In [1]:
import numpy as np
from pathlib import Path
from xrf.signatures.leaf_signature import Leaf_Signature_Extractor
from xrf.spatial.spatial_analyzer import Spatial_Analyzer
from xrf.config import Leaf_Signature_Config

In [2]:
# Project directory and data paths
PROJECT_DIR = Path.cwd()

# Find project root directory
while not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent

# Data path
# Load raw data

DATA_DIR = PROJECT_DIR / "data" / "xrf"
RAW_DATA_DIR = DATA_DIR / "raw"
OUTPUT_DATA_DIR = DATA_DIR / "output"
FIGURES_DIR = OUTPUT_DATA_DIR / "figures"
DIAG_DIR = OUTPUT_DATA_DIR / "diagnostics"
PROCESSED_DATA_DIR = OUTPUT_DATA_DIR / "processed"

# CHECK IF FIGURES_DIR EXISTS, IF NOT CREATE IT
if not FIGURES_DIR.exists():
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import json

sig_config = Leaf_Signature_Config()

# 1. Cargar dependencias del cuaderno 03
labels = np.load(PROCESSED_DATA_DIR / "page_001_labels.npy")
class_map = np.load(PROCESSED_DATA_DIR / "page_001_class_map.npy")
with open(PROCESSED_DATA_DIR / "page_001_meta.json", "r") as f:
    meta = json.load(f)
    optimal_k = meta["optimal_k"]

# 2. Abundancias
abundances = Leaf_Signature_Extractor.Compute_Abundances(labels, Num_Classes=optimal_k)
print("Vector de Abundancias (A_k):", np.round(abundances, 3))

# 3. Descriptores Espaciales (CCA)
spatial_features = []
for k in range(optimal_k):
    desc = Spatial_Analyzer.Extract_Spatial_Descriptors(
        class_map, Target_Class=k, Min_Size=sig_config.Min_Region_Size
    )
    spatial_features.append(desc)

print("Descriptores Espaciales (Clase 0):", spatial_features[0])

# 4. Ponderación de Documento Múltiple (Simulación de 5 páginas)
mock_signatures = np.random.rand(5, optimal_k)  # 5 páginas, K características
mock_weights = np.array([1.0, 0.8, 0.9, 0.2, 1.0])  # Página 4 tuvo mala calidad

global_signature = Leaf_Signature_Extractor.Compute_Weighted_Book_Signature(
    mock_signatures, mock_weights
)
print("Firma Global Ponderada (F_bar):", np.round(global_signature, 3))

# Exportar firma final
np.save(OUTPUT_DATA_DIR / "book_global_signature.npy", global_signature)

Vector de Abundancias (A_k): [0.301 0.192 0.095 0.097 0.062 0.085 0.064 0.103]
Descriptores Espaciales (Clase 0): {'Num_Regiones': 669.0, 'Tamano_Promedio': 120.40209267563527}
Firma Global Ponderada (F_bar): [0.353 0.645 0.418 0.442 0.456 0.62  0.516 0.519]
